# 01 · Datos

Carga de los 9 CSV, conversión a Parquet, incrementos de contadores por Δt y proporciones de bins de histogramas.

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import utils

pd.set_option('display.max_columns', 50)

%load_ext autoreload
%autoreload 2

In [ ]:
ro = utils.load_raw('train', 'operational_readouts')
counters, hists = utils.feature_groups(ro)
cols = counters + sum(hists.values(), [])

print('Forma:', ro.shape, '| vehículos:', ro[utils.ID].nunique())
print('Contadores:', counters)
print('Histogramas:', {k: len(v) for k, v in hists.items()})
print('Columnas sin clasificar:', [c for c in ro.columns if c not in cols + [utils.ID, utils.T]])

ro = ro.sort_values([utils.ID, utils.T])
g = ro.groupby(utils.ID)

print('\nLecturas por vehículo:\n', g.size().describe().round(1))
print('\nΔt entre lecturas:\n', g[utils.T].diff().describe().round(2))
print('\n% faltantes (top 10):\n', ro.isna().mean().mul(100).sort_values(ascending=False).head(50).round(2))

# ¿Son realmente acumulados? % de pasos donde la variable disminuye
dec = g[cols].diff().lt(0).mean().mul(100)
print('\n% de pasos con disminución (top 10):\n', dec.sort_values(ascending=False).head(10).round(3))

for k in ['specifications', 'tte']:
    df = utils.load_raw('train', k)
    print(f'\n{k}:', df.shape)
    print(df.head(3))

In [ ]:
for split in utils.SPLITS:
    ro = utils.load_raw(split, 'operational_readouts')
    ro = utils.add_increments(ro)
    utils.to_parquet(ro, f'{split}_readouts')
    utils.to_parquet(utils.load_raw(split, 'specifications'), f'{split}_specifications')
    print(split, ro.shape)